# SRQS8F7JWA9MZ

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import gc
from pathlib import Path
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from tools.filter import FilterDF as fdf
from tools.benchmarks import ParetoAnalysis as pa
from tools.benchmarks import AccuracyCalculation as ac
from tools.integrity_fixes import DataFixer as fix, DataExporter as exporter
from tools.coverage_functions import plot_time_series

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Read data from parquet files

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
    %store static_data_merged
    
# Data already exists
else:
    static_data = static_data_merged.copy()

# 

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
        
    %store sales_data_merged
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

# 

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
%store -r before_after_details_true
if 'before_after_details_true' not in locals():
    before_after_details_true = pd.read_csv('data/before_after_details_true.csv', index_col='location_id')
    %store before_after_details_true

# Timezones
%store -r timezones
if 'timezones' not in locals():
    timezones = pd.read_csv('data/timezones.csv', index_col='location_id')['timezone'].to_dict()
    for loc_id, df in sales_and_menu_data.items():
        df.index = df.index.tz_convert(timezones[loc_id])
        sales_and_menu_data[loc_id] = df
    %store timezones

%store -r restaurants_by_4m_coverage
if 'restaurants_by_4m_coverage' not in locals():
    restaurants_by_4m_coverage = pd.read_csv('data/2_palate_data_parquet_cleaned/restaurants_by_4m_coverage.csv')['location_id'].tolist()
    %store restaurants_by_4m_coverage

loc_id = 'SRQS8F7JWA9MZ'
df_uncleaned = sales_and_menu_data[loc_id]

locations = list(sales_and_menu_data.keys())
for other_loc_id in locations:
    if other_loc_id != loc_id:
        del sales_and_menu_data[other_loc_id]
        del sales_data_merged[other_loc_id]
gc.collect()

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.query('item_name.str.contains("Impossible")')['item_quantity'].sum()

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Impossible")')['item_quantity'].sum()

In [ ]:
plot_time_series('SRQS8F7JWA9MZ', 
                 df_uncleaned, 
                 before_after_details_true, 
                 freq='W', 
                 subset=True)
plt.show()

In [ ]:
plot_time_series('SRQS8F7JWA9MZ', 
                 df_uncleaned.query('item_name.str.contains("Impossible")'), 
                 before_after_details_true, 
                 freq='W', 
                 subset=False)
plt.show()

Time Differences

In [ ]:
time_differences_details[loc_id]

10 Hour Difference

In [ ]:
df_uncleaned.loc[pd.Timestamp('2019-07-17 4:45:58+00:00'):pd.Timestamp('2019-07-18').tz_localize('UTC')].head(5)

20 Hour Difference

In [ ]:
#rare_items = df_uncleaned["item_name"].value_counts().nsmallest(20).index.tolist()
#print(df.query('item_name == "Cold Brew Coffee ‚Òïô∏È"', engine='python')[['item_name','item_modifications']].value_counts().to_string())
# print(df.query('item_name in @rare_items', engine='python')[['item_name','item_modifications']].value_counts().to_string())

In [ ]:
# Item names to consolidate
name_changes = {"Impossible Patty Melt" : ["Impossible Melt"],
              "Gold Standard - Impossible" : ["Gold Standard Impossible", "Imp Breakfast"], 
              "Gold Standard - Bacon" : ["Gold Standard - Bacon!", "Gold Standard Breakfast Sandwich"],
              "Gold Standard - Bacon & Kale" : ["Gold Standard - Bacon/Kale", "Gold Standard - Bacon + Kale", "Both - Gold Standard", "Gold Standard - Bacon Ü•Ì"],  
              "Telway Burger" : ["Telway"], 
              "The Alternative" : ["Alternative"],
              "Beyond Burger" : ["Beyond Burg"],
              "Bulldog Butty" : ["Butty"],
              "Four Guys Burger" : ["Four Guys", "4 Guys", "Guys"],
              "Iced Coffee" : ["Cold Brew Coffee", "Cold Brew Coffee ‚Òïô∏È"],
              "Tea" : ["Ice T"],
              "Mexican Coca-Cola": ["Mexi-Coke"],
              "Miss Vickie" : ["Miss Vickie‚Äôs"],
              "Hot Dog": ["Coney Island Dog"]}

# Swap the keys and values
name_changes_dict = {variant: canonical for canonical, variants in name_changes.items() for variant in variants}

# Item names to swap based on modications
modification_name_changes = [('Gold Standard - Impossible', 'Add Bacon|Extra Bacon|Regular Bacon|Beef', 'Gold Standard - Bacon/Beef & Impossible'),
                             ('Beyond Burger', 'Add Bacon|Beef', 'Bacon/Beef Beyond Burger'),
                             ('Beyond Burger', 'No Cheese|No Chz|No C|No Cuz|Vegan', 'Vegan Beyond Burger'),
                             ('Impossible Patty Melt', 'Add Bacon|Beef|Meat', 'Bacon/Beef Impossible Patty Melt'),
                             ('Impossible Patty Melt', 'No Cheese|No Chz|No C|No Cuz|Vegan', 'Vegan Impossible Patty Melt'),
                             ('Gold Standard - Bacon|Gold Standard - Bacon & Kale', 'Sub Impossible', 'Gold Standard - Impossible'),
                             ('Gold Standard - Bacon|Gold Standard - Bacon & Kale', 'Add Impossible', 'Gold Standard - Bacon & Impossible'),
                             ('Gold Standard - Bacon|Gold Standard - Bacon & Kale', 'No Bacon', 'Gold Standard - Kale'),
                             ('Gold Standard - Kale', 'Bacon And Kale', 'Gold Standard - Bacon & Kale'),
                             ('Gold Standard - Kale', 'Impossible', 'Gold Standard - Impossible'),
                             ('Gold Standard Sandwich', 'Bacon', 'Gold Standard - Bacon'),
                             ('Gold Standard Sandwich', 'Impossible', 'Gold Standard - Impossible'),
                             ('Gold Standard Sandwich', '', 'Gold Standard - Kale'),
                             ('The Alternative', 'Bacon|Beef', 'Bacon/Beef The Alternative'),
                             ('The Alternative', 'Egg|Cheese', 'Egg/Cheese The Alternative')]

# Turn into dataframe for viewing
modification_name_changes_df = pd.DataFrame(data = modification_name_changes, columns = ['name', 'modification', 'new_name'])

alcoholic_drinks = []

non_alcoholic_drinks = [
    "Juice",
    "Coca-Cola",
    "Coconut Water",
    "Mexican Coca-Cola",
    "La Croix",
    "Water",
    "Ginger Beer",
    "Cheerwine",
    "Mata Mate",
    "Mate Libre",
    "Zamalek",
    "Crodino",
    "Lemonade",
    "Club Mate",
    "Tea",
    "Cream Soda",
    "Mate",
    "Laura Palmer"
]

merch_list = [
    "Egift Card",
]

rare_list = [
    "Laura Palmer",
    "Donut",
    "Cool Ranch"
]

condiment_list = [
    "Ketchup Pack",
    "Marie Sharp'S"
]

unknown_list = [
    "Og FaveÜç¶",
    "Mom Jeans",
    "Cool",
    "Hi",
    "Kids Thing $",
    "Blah",
    "$"
]

remove_list = merch_list + unknown_list + condiment_list

half_vegan_list = [
    'Coffee',
    'Iced Coffee'
]

vegetarian_list = [
    'Gold Standard - Kale', 
    'Gold Standard - Impossible', 
    'Impossible Patty Melt', 
    'Beyond Burger', 
    'Cole Slaw', 
    'Miss Vickie', 
    'Ice Cream Sandwich',
    'Eggnog',
    'Donut',
    'Egg/Cheese The Alternative'
]

vegan_list = [
    "The Alternative",
    "Vegan Beyond Burger",
    "Vegan Impossible Patty Melt",
    "Ketchup Pack",
    "Marie Sharp'S"
]

items_to_remove = []

In [ ]:
# Bacon or Bacon & Kale
# No Bacon -> vegetarian
# No Cheese -> carnist
# No Cheese, No Bacon, No Aioli -> vegan

# Kale
# Add Impossible -> Impossible
# 

# Impossible
# Add Bacon -> carnist
#

# Bacon & Impossible

animal_product_conditions = ['item_modifications.str.contains("Bacon")',
                             'item_modifications.str.contains("Cheese")',
                             'item_modifications.str.contains("Impossible")',
                             'item_modifications.str.contains("Aioli")',
                             'item_modifications.str.contains("Egg")',
                             'item_modifications.str.contains("Cheddar")',
                             'item_modifications.str.contains("Sub")'
                             ]
print(df_uncleaned.query('item_name.str.contains("Beyond") and (' + ' or '.join(animal_product_conditions) + ')')[['item_name','item_modifications']].value_counts().to_string())

In [ ]:
# items_tagged_cleaned = (items_tagged
#                         .query('location_id == @loc_id')
#                         .assign(item_name=lambda df: df['item_name']
#                                 .str.strip('123456789./\\ ')  # Clean up item names
#                                 .replace(replacement_dict)    # Replace names based on dictionary
#                       #.replace(items_to_remove, pd.NA))  # Replace non-dish items with NA
#                       #.dropna(subset=['item_name']
#                       )
#                       .assign(dish_category = lambda df: df['dish_category'].mask(df['item_name'].isin(non_alcoholic_drinks), 'Drink'))
#                       .assign(dish_category = lambda df: df['dish_category'].mask(df['dish_category'].eq('Water'), 'Drink'))
#                       .assign(dish_category = lambda df: df['dish_category'].mask(df['item_name'].isin(merch), 'Merch'))
#                       .drop_duplicates('item_name'))

In [ ]:
df = (df_uncleaned
              .assign(item_name = lambda df: df['item_name']
                      .str.strip('123456789./\\ ')  # Clean up item names
                      .replace(name_changes_dict),    # Consolidate dishes
                      )
              .assign(item_name = lambda df: np.select(condlist = [df['item_name'].eq(name) &  
                                                                   df['item_modifications'].str.contains(modification) for name, modification, _ in modification_name_changes],
                                                       choicelist = modification_name_changes_df['new_name'].tolist(),
                                                       default = df['item_name']),
                      dish_category = lambda df: 
                              df['dish_category']
                              .mask(df['item_name'].isin(non_alcoholic_drinks), 'Drink')
                              .mask(df['item_name'].isin(unknown_list), 'Unknown')
                              .mask(df['item_name'].isin(merch_list), 'Merch'),
                      vegetarian = lambda df: 
                              df['item_name']
                              .isin(vegetarian_list + vegan_list + non_alcoholic_drinks),
                      vegan = lambda df: 
                              df['item_name']
                              .isin(vegan_list + non_alcoholic_drinks)
                              .mask(df['item_name'].isin(half_vegan_list), np.random.rand(len(df)) < 0.5)
                      )
              .query('~item_name.isin(@remove_list)') # Important: do we actually want to remove unknowns
              #.drop('unique_id', axis=1)
             )
food_df = df.query('~dish_category.isin(["Alcohol", "Drink", "Merch", "Rare"])')

In [ ]:
166/188


In [ ]:
food_df.shape

In [ ]:
(df_menu
                 ['item_name']
                 .value_counts()
                 .to_frame(name='c'))

In [ ]:
to_remove = ['Miss Vickie', 'Egg Nog', 'Hot Dog', 'Pork Sam']

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import FuncFormatter

In [ ]:
# Visualizing with gaps for inactive weeks
introduction_fig, ax = plt.subplots(figsize=(14, 8))

# Index into the promotional items for this restaurant
promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert('UTC')

top_n = 20

unique_dishes = (df_menu.query('~item_name.isin(@to_remove)')
                 ['item_name']
                 .value_counts()
                 .to_frame(name='c')
                # [:top_n]
                 .index[::-1]
                 )

legend_handles = []

for dish in unique_dishes:
    
    #print(dish)
    
    dish_df = df_menu.query('item_name == @dish')
    
    vmin = dish_df['unit_price'].min()
    vmax = dish_df['unit_price'].max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.ScalarMappable(norm=norm, cmap='magma')
    
    weekly_quantities = (dish_df
                         .resample('W')
                         .agg({'item_quantity': 'sum', 'unit_price': 'mean'})
                         .query('0 < item_quantity')
                         .assign(week = lambda df: df.index.tz_localize(None).to_period('W'))
                         .set_index('week')
                         )
    
    # For every active week
    for week, row  in weekly_quantities.iterrows():
        
        weekly_quantity = row['item_quantity']
        dot_size = weekly_quantity/50 + 2.5
        weekly_price = row['unit_price']
        color = cmap.to_rgba(weekly_price)

        # Place a blue dot
        ax.hlines(y=dish, xmin=week.start_time, xmax=week.end_time, colors=color, lw=dot_size, label=loc_id)
        
    ax.text(x=food_df.index[-1] + pd.DateOffset(100), y=dish, s=f'${vmin/100:.2f}-${vmax/100:.2f}', verticalalignment='center', horizontalalignment='left', fontsize='x-small', color='gray')
    
    # Create a custom legend entry for this dish
    #color_patch_min = mpatches.Patch(color=cmap.to_rgba(vmin), label=f'{dish} Min: ${vmin/100:.2f}')
    #color_patch_max = mpatches.Patch(color=cmap.to_rgba(vmax), label=f'{dish} Max: ${vmax/100:.2f}')
    #legend_handles.extend([color_patch_min, color_patch_max])

# Draw a vertical red line at the promo date
ax.axvline(x=promo_datetime, color='red', linestyle='--', linewidth=1, alpha=0.5)


global_min = 5.31
global_max = 12.75
norm_global = mcolors.Normalize(vmin=global_min, vmax=global_max)
sm = cm.ScalarMappable(norm=norm_global, cmap='magma')
sm.set_array([])

base_pos = [0.88, 0.1, 0.02, 0.3]

# Calculate the normalized shift for a desired shift in inches.
# Figure width in inches:
fig_width = introduction_fig.get_size_inches()[0]
shift_inches = 2.5  # change this value for a different shift
normalized_shift = shift_inches / fig_width

# Shift the colorbar to the left by subtracting the normalized shift from the x coordinate.
adjusted_pos = [base_pos[0] - normalized_shift, base_pos[1], base_pos[2], base_pos[3]]

# Create the colorbar axis at the adjusted position.
cbar_ax = introduction_fig.add_axes(adjusted_pos)
cbar = plt.colorbar(sm, cax=cbar_ax, orientation='vertical')

# Set a custom tick formatter to show ticks divided by 100 (as dollars)
def dollar_formatter(x, pos):
    return f'${x:.2f}'

cbar.ax.yaxis.set_major_formatter(FuncFormatter(dollar_formatter))
# Set the colorbar "title"
cbar.set_label("Gold Standard Impossible")


# Plot
ax.set_title(f'Weekly Sales of Top {top_n} Dishes for {loc_id}')
ax.set_xlabel('Date')
ax.set_ylabel('Dish')
#ax.legend(handles=legend_handles, title="Price Range per Dish", fontsize='small', loc='upper left', bbox_to_anchor=(1, 1))

# Figure
introduction_fig.tight_layout(rect=[0, 0, 0.85, 1])

plt.show()

In [ ]:
df_menu['item_name'].value_counts()

In [ ]:
col = 'dish_category'
multi_categorized = df.groupby('item_name')[col].nunique().to_frame(name='c').query('c > 1').index
df.groupby('item_name')[col].unique().to_frame(name='c').query('item_name in @multi_categorized')

In [ ]:
df.groupby('item_name')['vegan'].unique()

In [ ]:
df.value_counts('item_name').filter(regex='Gold Standard').index.tolist(),

In [ ]:
df.value_counts('item_name').filter(regex='Gold Standard').index.tolist()

In [ ]:
menu_changes = {
    'Gold Standard Breakfast': ['Gold Standard - Bacon',
 'Gold Standard - Kale',
 'Gold Standard - Bacon + Kale',
 'Gold Standard - Bacon \uf8ffÜ•Ì',
 'Gold Standard - Bacon/Kale',
 'Gold Standard Sandwich',
 'Both - Gold Standard',
 'Gold Standard - Bacon & Kale',
 'Gold Standard Breakfast Sandwich',
 'Gold Standard - Bacon!'],
    'Gold Standard Impossible': ['Gold Standard - Impossible', 'Gold Standard - Bacon/Beef & Impossible'],
    'Beyond Burger': df.value_counts('item_name').filter(regex='Beyond').index.tolist(),
    'Bottled Pop': df.value_counts('item_name').filter(regex='Coca|Crod|Ginger|Soda').index.tolist(),
    'Canned Drinks': df.value_counts('item_name').filter(regex='Coco|Mate|Zam|Croi').index.tolist(),
    'The Alternative': df.value_counts('item_name').filter(regex='Alternative').index.tolist(),
    'Impossible Patty Melt': df.value_counts('item_name').filter(regex='Patty Melt').index.tolist()
    }

# Swap the keys and values
menu_dish_dict = {variant: canonical for canonical, variants in menu_changes.items() for variant in variants}

df_menu = (df
           .assign(item_name = lambda df: df['item_name']
                      .replace(menu_dish_dict),    # Consolidate dishes
                      )
           .query('~item_name.isin(@rare_list)'))

In [ ]:
df_menu['item_name'].value_counts()

In [ ]:
food_df.query('item_type != "Drink" and dish_category != "Alcohol"').shape

In [ ]:
# df1 = df.copy()
# df1_menu = df_menu.copy()
# %store df1
# %store df1_menu
%store -r df1
%store -r df1_menu

In [ ]:
print(df1
      .query('~vegetarian')
      ['unit_price']
      .mean())

print((df1
       .query('~vegetarian')
       ['item_name']
       .nunique()) / (df1
                      ['item_name']
                      .nunique()))
plt.plot(df1
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['unit_price']
         .mean(), 
         df1
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"').resample('W')['item_name'].nunique() / df1.query('dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['item_name']
         .nunique(), 
         'o', 
         alpha=0.5)
plt.show()

In [ ]:
df_menu['item_name'].value_counts().to_frame()

In [ ]:
food_df.query('item_name.str.contains("Alt") and item_modifications.str.contains("Impossible")')['item_quantity'].sum()

In [ ]:
food_df['item_name'].value_counts().to_frame()

In [ ]:
df1['item_name'].value_counts().to_frame()

Analysis 1

In [ ]:
# # Covariate creation
# lookback_period = 1
# lookback_unit = 'D'

# def season_from_month(month):
#     return 'winter' if month in [12, 1, 2] else \
#            'spring' if month in [3, 4, 5] else \
#            'summer' if month in [6, 7, 8] else 'fall'

# def weighted_avg(window):
#     return (window['item_quantity'] * window['unit_price']).sum() /window['item_quantity'].sum()

# hour_mapping = {
#     22: -1, 23: -1, 
#     1: -1, 6: -1, 7: -1
# }

# df1 = df.copy()

# df1 = df1.query('item_type != "Drink"')

# model_data = (df1
#               .assign(
#                   hour_of_day = lambda df: df.index.to_series().dt.hour.replace(hour_mapping).astype("category"),
#                   day_of_week = lambda df: df.index.to_series().dt.dayofweek.astype("category"),
#                   weekend = lambda df: pd.Series(df.index.dayofweek.isin([5, 6]).astype(int), index=df.index).astype("category"),
#                   meal_period = lambda df: pd.cut(df.index.to_series().dt.hour.astype("category"), 
#                                     bins=[0, 5, 11, 16, 22, 24], 
#                                     labels=['Late', 'Breakfast', 'Lunch', 'Dinner', 'Late'], 
#                                     right=False,
#                                     ordered=False).replace({'Late': 'Dinner'}),
#                   day_of_month = lambda df: df.index.to_series().dt.day.astype("category"),
#                   month = lambda df: df.index.to_series().dt.month.astype("category"),
#                   season = lambda df: df.index.month.map(season_from_month).astype("category"),
#                   date = lambda df: df.index.to_series().dt.date.astype("category").cat.codes)
#               .reset_index()
#               .set_index('unique_id')
#               .join([(df1
#                       .query('~vegetarian')
#                       ['item_price']            
#                       .rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df1.query('~vegetarian')['unique_id'].values, axis=0)
#                       .rename('meat_window_price')),
#                      (df1
#                       .query('~vegetarian')
#                       ['item_quantity'].rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df1.query('~vegetarian')['unique_id'].values, axis=0)
#                       .rename('meat_window_quantity')),
#                      (df1
#                       .query('vegetarian')
#                       ['item_price'] 
#                       .rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df1.query('vegetarian')['unique_id'].values, axis=0)
#                       .rename('vegetarian_window_price')),
#                      (df1
#                       .query('vegetarian')
#                       ['item_quantity']
#                       .rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df1.query('vegetarian')['unique_id'].values, axis=0)
#                       .rename('vegetarian_window_quantity')),
#                      (df1
#                       .query('vegan')
#                       ['item_price'] 
#                       .rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df1.query('vegan')['unique_id'].values, axis=0)
#                       .rename('vegan_window_price')),
#                      (df1
#                       .query('vegan')
#                       ['item_quantity'] 
#                       .rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df1.query('vegan')['unique_id'].values, axis=0)
#                       .rename('vegan_window_quantity'))
#                     ],
#                     how='left')
#               .reset_index()
#               .set_index('created_at')
#               .assign(
#                   vegan_window_price = lambda df: df['vegan_window_price'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   vegan_window_quantity = lambda df: df['vegan_window_quantity'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   vegetarian_window_price = lambda df: df['vegetarian_window_price'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   vegetarian_window_quantity = lambda df: df['vegetarian_window_quantity'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   meat_window_price = lambda df: df['meat_window_price'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   meat_window_quantity = lambda df: df['meat_window_quantity'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   vegan_window_avg = lambda df: df['vegan_window_price'] / df['vegan_window_quantity'],
#                   vegetarian_window_avg = lambda df: df['vegetarian_window_price'] / df['vegetarian_window_quantity'],
#                   meat_window_avg = lambda df: df['meat_window_price'] / df['meat_window_quantity'],
#                   vegan_outcome = lambda df: 1*df['vegan'],
#                   vegetarian_outcome = lambda df: 1*df['vegetarian']
#               ))

# model_data = model_data.loc[model_data.index.repeat(model_data['item_quantity'])]
# model_data['item_quantity'] = 1


# # majority = model_data.query('~vegan')
# # minority = model_data.query('vegan')

# # from sklearn.utils import resample

# # majority_downsampled = resample(majority, 
# #                                 replace=False, 
# #                                 n_samples=len(majority),
# #                                 random_state=1)
# # balanced_data = pd.concat([majority_downsampled, minority])

# model_data['vegan'].value_counts()

In [ ]:
# interaction_predictors = [
#     'meat_window_avg',
#     'vegetarian_window_avg',
#     'vegan_window_avg',
# ]

# time_predictors = [
#     'hour_of_day',
#     'meal_period',
#     'weekend',
#     'day_of_week',
#     'day_of_month',
#     'month',
#     'season',
#     #'date',
# ]

# # Split the data into training and testing sets
# train_size = model_data.shape[0] // 2

# train_data = (model_data
#               .dropna(subset=interaction_predictors + time_predictors)
#               .reset_index()
#               .iloc[:train_size, :])
# test_data = (model_data
#              .dropna(subset=interaction_predictors + time_predictors)
#              .reset_index()
#              .iloc[train_size:, :])

# # Fit logistic regression model
# formula = ['vegan_outcome ~ ',
#            # '(',' + '.join(interaction_predictors),')**2',
#            # ' + ',
#            ' + '.join(interaction_predictors), 
#            ' + vegan_window_avg:meat_window_avg + vegetarian_window_avg:meat_window_avg',
#            ' + ',
#            ' + '.join(time_predictors)
#            ]
# logit_model = smf.logit(''.join(formula), train_data)
# logit_fit = logit_model.fit(maxiter=200)

# # Fit ARIMA model to residuals
# from statsmodels.tsa.arima.model import ARIMA
# train_data['pred'] = logit_fit.predict(train_data)
# train_data['residuals'] = train_data['vegan_outcome'] - train_data['pred']
# arima_model = ARIMA(train_data['residuals'], order=(1, 0, 1)).fit()
# # print(arima_model.summary())
# # print(logit_fit.summary())

# # Predict for training and testing sets
# train_data['pred'] = logit_fit.predict(train_data)
# train_data['arima_adjusted_pred'] = arima_model.fittedvalues + train_data['pred']
# test_data['pred'] = logit_fit.predict(test_data)
# test_data['arima_forecast'] = arima_model.get_forecast(steps=len(test_data)).predicted_mean
# test_data['arima_adjusted_pred'] = test_data['pred'] + test_data['arima_forecast']

# # Plot function for resampling and visualization
# def plot_resampled(data, freq, start_date=None, end_date=None, title="Resampled Predictions"):
#     resampled_pred = data.resample(freq)['arima_adjusted_pred'].mean()
#     resampled_actual = data.resample(freq)['vegan_outcome'].mean()

#     if start_date and end_date:
#         resampled_pred = resampled_pred.loc[start_date:end_date]
#         resampled_actual = resampled_actual.loc[start_date:end_date]

#     resampled_pred.plot(color='orange', label='Predicted')
#     resampled_actual.plot(color='blue', alpha=0.3, label='Actual')

#     plt.title(title)
#     plt.legend()
#     plt.show()

# # Resample and plot results for training data
# train_result = train_data.set_index('created_at')
# plot_resampled(train_result, '7D', title="Training Data: Weekly Resampled Predictions")
# plot_resampled(train_result, '7D', start_date='2020', end_date='2021', title="Training Data: Weekly (2020-2021)")
# plot_resampled(train_result, '30D', title="Training Data: Monthly Resampled Predictions")

# # Resample and plot results for testing data
# test_result = test_data.set_index('created_at')
# plot_resampled(test_result, '7D', title="Testing Data: Weekly Resampled Predictions")
# plot_resampled(test_result, '7D', start_date='2020', end_date='2021', title="Testing Data: Weekly (2020-2021)")
# plot_resampled(test_result, '30D', title="Testing Data: Monthly Resampled Predictions")

